In [ ]:
import os, json, time, copy, glob, shutil
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision
from PIL import Image
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import matplotlib
TEST = bool(os.environ.get("SWEEP_TEST"))      
if TEST:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt

DATA_ROOT = "/kaggle/input/data"     
OUT = "/kaggle/working"
IMG = 224
SKEWS = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]   
SEEDS = [0, 1, 2]
N_POS = 500                  
NEG_POOL_PER_BRACKET = 2000  
EPOCHS = 8
BATCH = 32
LR = 1e-4
PRETRAINED = True            
MARKER_FRAC = 0.9            
MARK = (188, 220, 8, 40)     
SPLIT_SEED = 42

if TEST:
    DATA_ROOT, OUT = os.environ["SWEEP_DATA"], os.environ["SWEEP_OUT"]
    SKEWS, SEEDS, N_POS, NEG_POOL_PER_BRACKET = [0.5, 0.9], [0], 16, 40
    EPOCHS, BATCH, PRETRAINED = 1, 8, False

if not os.path.exists(os.path.join(DATA_ROOT, "Data_Entry_2017.csv")):          
    hits = glob.glob("/kaggle/input/**/Data_Entry_2017.csv", recursive=True)
    assert hits, "NIH dataset not found: use 'Add Data' on the right and add 'NIH Chest X-rays', then re-run."
    DATA_ROOT = os.path.dirname(hits[0])
print("DATA_ROOT:", DATA_ROOT)

for sub in ("runs", "models", "figures"):
    os.makedirs(os.path.join(OUT, sub), exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| torch", torch.__version__)
if device == "cpu" and not TEST:
    print("WARNING: no GPU found. Notebook settings -> Accelerator -> GPU, then restart.")

In [ ]:
def age_bracket(a):
    return "young" if a <= 40 else ("old" if a >= 60 else "mid")

meta = pd.read_csv(os.path.join(DATA_ROOT, "Data_Entry_2017.csv"))
meta = meta[(meta["Patient Age"] > 0) & (meta["Patient Age"] < 100)].copy()
is_pos = meta["Finding Labels"].str.contains("Cardiomegaly")
meta = meta[is_pos | (meta["Finding Labels"] == "No Finding")].copy()
meta["label"] = meta["Finding Labels"].str.contains("Cardiomegaly").astype(int)
meta["age_bracket"] = meta["Patient Age"].apply(age_bracket)
meta = meta[meta.age_bracket != "mid"]

rng = np.random.default_rng(SPLIT_SEED)
patients = meta["Patient ID"].unique()
rng.shuffle(patients)
n_val = int(len(patients) * 0.15)
val_p, test_p = set(patients[:n_val]), set(patients[n_val:2 * n_val])
meta["split"] = np.where(meta["Patient ID"].isin(val_p), "val",
                         np.where(meta["Patient ID"].isin(test_p), "test", "train"))
tr, va, te = (meta[meta.split == s] for s in ("train", "val", "test"))

def cell(df, lab, br):
    return df[(df.label == lab) & (df.age_bracket == br)]

def mix(pool_old, pool_young, n, old_frac, seed):
    n_old = int(round(n * old_frac))
    return pd.concat([pool_old.sample(n=n_old, random_state=seed),
                      pool_young.sample(n=n - n_old, random_state=seed)])

P_old, P_young = cell(tr, 1, "old"), cell(tr, 1, "young")
print(f"train positives: old={len(P_old)}  young={len(P_young)}")

n_pos = min(N_POS, int(len(P_old) / max(SKEWS)), 2 * len(P_young))
n_pos -= n_pos % 2
n_neg = 2 * n_pos
print(f"every training set: {n_pos} positives + {n_neg} negatives")

NEG_OLD = cell(tr, 0, "old").sample(n=NEG_POOL_PER_BRACKET, random_state=0)
NEG_YOUNG = cell(tr, 0, "young").sample(n=NEG_POOL_PER_BRACKET, random_state=0)

def make_train(f, seed):
    rs = seed * 1000 + int(round(f * 100))
    pos = mix(P_old, P_young, n_pos, f, rs)
    neg = mix(NEG_OLD, NEG_YOUNG, n_neg, 1 - f, rs)
    return pd.concat([pos, neg]).sample(frac=1, random_state=rs).reset_index(drop=True)

TP_old, TP_young = cell(te, 1, "old"), cell(te, 1, "young")
def max_n(po, py, old_frac):
    return int(min(len(po) / old_frac, len(py) / (1 - old_frac)))
n_te = min(max_n(TP_old, TP_young, 0.5), max_n(TP_old, TP_young, 0.1))
n_te -= n_te % 2

def make_test(pos_old_frac, neg_old_frac, seed=7):
    pos = mix(TP_old, TP_young, n_te, pos_old_frac, seed)
    neg = mix(cell(te, 0, "old"), cell(te, 0, "young"), 2 * n_te, neg_old_frac, seed)
    return pd.concat([pos, neg]).reset_index(drop=True)

test_bal = make_test(0.5, 0.5)    
test_inv = make_test(0.1, 0.9)    

VP = va[va.label == 1]
VN = va[va.label == 0].sample(n=len(VP), random_state=0)
val_small = pd.concat([VP, VN]).sample(frac=1, random_state=0).reset_index(drop=True)

names = sorted(pd.concat([P_old, P_young, NEG_OLD, NEG_YOUNG, val_small, test_bal, test_inv])["Image Index"].unique())
print(f"test sets: {len(test_bal)} images each | val: {len(val_small)} | images to load: {len(names)}")
print(pd.crosstab(test_inv.label, test_inv.age_bracket))

In [ ]:
cache_path, names_path = os.path.join(OUT, "cache_u8.npy"), os.path.join(OUT, "cache_names.json")
if os.path.exists(cache_path) and os.path.exists(names_path) and json.load(open(names_path)) == names:
    arr = np.load(cache_path)
    print("loaded cache from disk:", arr.shape)
else:
    t0 = time.time()
    print("indexing PNG files (about a minute)...")
    paths = {os.path.basename(p): p for p in glob.glob(os.path.join(DATA_ROOT, "**", "*.png"), recursive=True)}
    missing = [n for n in names if n not in paths]
    assert not missing, f"{len(missing)} images not found under {DATA_ROOT}, e.g. {missing[:3]}"
    BILINEAR = getattr(Image, "Resampling", Image).BILINEAR
    def load_one(n):
        return np.asarray(Image.open(paths[n]).convert("L").resize((IMG, IMG), BILINEAR))
    with ThreadPoolExecutor(max_workers=8) as ex:
        arr = np.stack(list(ex.map(load_one, names)))
    np.save(cache_path, arr)
    json.dump(names, open(names_path, "w"))
    print(f"loaded {arr.shape} in {time.time() - t0:.0f}s")
cache = torch.from_numpy(arr).to(device)          
row_of = {n: i for i, n in enumerate(names)}

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
R0, R1, C0, C1 = MARK
use_amp = (device == "cuda")

def to_idx(df):
    return torch.tensor([row_of[n] for n in df["Image Index"]], device=device)

def auc(y, p):
    return float(roc_auc_score(y, p))

def make_batch(idx, train=False, marker=None):
    x = cache[idx]
    if train:                                           
        flip = torch.rand(len(idx), device=device) < 0.5
        x = torch.where(flip.view(-1, 1, 1), x.flip(-1), x)
    x = x.float().div(255.0)
    if marker is not None:
        x[marker, R0:R1, C0:C1] = 1.0
    x = x.unsqueeze(1).expand(-1, 3, -1, -1)
    return (x - MEAN) / STD

def new_model():
    w = torchvision.models.DenseNet121_Weights.DEFAULT if PRETRAINED else None
    m = torchvision.models.densenet121(weights=w)
    m.classifier = nn.Linear(m.classifier.in_features, 1)
    return m.to(device)

def _mk(marker, sl):
    return None if marker is None else torch.as_tensor(marker[sl], device=device)

@torch.no_grad()
def forward_all(model, idx, marker=None, bs=128, feats=False):
    model.eval()
    out = []
    for i in range(0, len(idx), bs):
        sl = slice(i, i + bs)
        x = make_batch(idx[sl], marker=_mk(marker, sl))
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
            if feats:
                o = F.adaptive_avg_pool2d(F.relu(model.features(x)), 1).flatten(1)
            else:
                o = torch.sigmoid(model(x).squeeze(1).float())
        out.append(o.float().cpu())
    return torch.cat(out).numpy()

val_idx, val_y = to_idx(val_small), val_small["label"].values

def train_model(train_df, seed, marker=None, val_marker=None):
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = new_model()
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    try:
        scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    except Exception:                      
        scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    idx = to_idx(train_df)
    y = torch.tensor(train_df["label"].values, dtype=torch.float32, device=device)
    n, best, best_state = len(idx), -1.0, None
    for epoch in range(EPOCHS):
        model.train()
        perm = torch.randperm(n, device=device)
        for i in range(0, n, BATCH):
            b = perm[i:i + BATCH]
            if len(b) < 2:                               
                continue
            mk = None if marker is None else torch.as_tensor(marker[b.cpu().numpy()], device=device)
            x = make_batch(idx[b], train=True, marker=mk)
            with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
                logit = model(x).squeeze(1)
            loss = F.binary_cross_entropy_with_logits(logit.float(), y[b])
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
        v = auc(val_y, forward_all(model, val_idx, val_marker))
        if v > best:
            best, best_state = v, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return model, best

def age_gaps(df, prob):
    m = pd.Series(prob, index=df.index).groupby([df.label, df.age_bracket]).mean()
    return {"gap_healthy": float(m.get((0, "old"), np.nan) - m.get((0, "young"), np.nan)),
            "gap_diseased": float(m.get((1, "old"), np.nan) - m.get((1, "young"), np.nan))}

def fit_head(X, y):
    return make_pipeline(StandardScaler(),
                         LogisticRegressionCV(Cs=8, cv=5, scoring="roc_auc", max_iter=3000, random_state=0)).fit(X, y)

T_DF = {"test_balanced": test_bal, "test_inverted": test_inv}
T_IDX = {k: to_idx(v) for k, v in T_DF.items()}
print("helpers ready")

In [ ]:
_E = EPOCHS
EPOCHS = 1
t0 = time.time()
_m, _v = train_model(make_train(0.9, 0), 0)
per_epoch = time.time() - t0
EPOCHS = _E
n_runs = len(SKEWS) * len(SEEDS) + len(SEEDS)            
print(f"1 epoch took {per_epoch:.0f}s  ->  full sweep estimate: {per_epoch * EPOCHS * n_runs / 60:.0f} min "
      f"for {n_runs} training runs")
del _m

In [ ]:
def evaluate_run(model, f, seed, name, best_val, repair_df):
    res = {"name": name, "f": f, "seed": seed, "val": best_val}
    rows = []
    for t, d in T_DF.items():                                   
        p = forward_all(model, T_IDX[t])
        res[t] = auc(d.label.values, p)
        if t == "test_balanced":
            res.update(age_gaps(d, p))
        rows.append(d.assign(set=t, prob=p, method="finetuned"))
    head = fit_head(forward_all(model, to_idx(repair_df), feats=True), repair_df.label.values)
    for t, d in T_DF.items():
        p = head.predict_proba(forward_all(model, T_IDX[t], feats=True))[:, 1]
        res["repair_" + t] = auc(d.label.values, p)
        if t == "test_balanced":
            res.update({"repair_" + k: v for k, v in age_gaps(d, p).items()})
        rows.append(d.assign(set=t, prob=p, method="repaired"))
    return res, pd.concat(rows, ignore_index=True)

def run_age_sweep():
    for seed in SEEDS:
        repair_df = make_train(0.5, seed)
        for f in SKEWS:
            name = f"age_f{int(round(f * 100))}_s{seed}"
            path = os.path.join(OUT, "runs", name + ".json")
            if os.path.exists(path):
                print("skip (done):", name)
                continue
            t0 = time.time()
            model, best_val = train_model(make_train(f, seed), seed)
            res, preds = evaluate_run(model, f, seed, name, best_val, repair_df)
            preds.to_csv(os.path.join(OUT, "runs", name + "_preds.csv"), index=False)
            if f == 0.9 and seed == SEEDS[0]:
                torch.save(model.state_dict(), os.path.join(OUT, "models", name + ".pt"))
            json.dump(res, open(path, "w"), indent=1)
            print(f"{name}: val={res['val']:.3f}  bal={res['test_balanced']:.3f}  inv={res['test_inverted']:.3f}"
                  f"  | repaired bal={res['repair_test_balanced']:.3f} inv={res['repair_test_inverted']:.3f}"
                  f"  ({time.time() - t0:.0f}s)")

def run_imagenet_control():
    for seed in SEEDS:
        name = f"imagenet_s{seed}"
        path = os.path.join(OUT, "runs", name + ".json")
        if os.path.exists(path):
            continue
        torch.manual_seed(seed)
        model = new_model()
        rd = make_train(0.5, seed)
        head = fit_head(forward_all(model, to_idx(rd), feats=True), rd.label.values)
        res = {"name": name, "seed": seed}
        for t, d in T_DF.items():
            p = head.predict_proba(forward_all(model, T_IDX[t], feats=True))[:, 1]
            res[t] = auc(d.label.values, p)
            if t == "test_balanced":
                res.update(age_gaps(d, p))
        json.dump(res, open(path, "w"), indent=1)
        print(f"{name}: bal={res['test_balanced']:.3f}  inv={res['test_inverted']:.3f}")

run_age_sweep()
run_imagenet_control()

In [ ]:
def marker_flags(df, pos_rate, neg_rate, seed):
    r = np.random.default_rng(seed).random(len(df))
    return np.where(df.label.values == 1, r < pos_rate, r < neg_rate)

MK_BAL = marker_flags(test_bal, 0.5, 0.5, 301)                          
MK_INV = marker_flags(test_inv, 1 - MARKER_FRAC, MARKER_FRAC, 302)      

def run_marker():
    for seed in SEEDS:
        name = f"marker_s{seed}"
        path = os.path.join(OUT, "runs", name + ".json")
        if os.path.exists(path):
            print("skip (done):", name)
            continue
        t0 = time.time()
        trn = make_train(0.5, seed)
        model, best_val = train_model(trn, seed,
                                      marker=marker_flags(trn, MARKER_FRAC, 1 - MARKER_FRAC, 100 + seed),
                                      val_marker=marker_flags(val_small, MARKER_FRAC, 1 - MARKER_FRAC, 200 + seed))
        res = {"name": name, "seed": seed, "val": best_val,
               "marker_test_balanced": auc(test_bal.label.values, forward_all(model, T_IDX["test_balanced"], MK_BAL)),
               "marker_test_inverted": auc(test_inv.label.values, forward_all(model, T_IDX["test_inverted"], MK_INV)),
               "clean_test_balanced": auc(test_bal.label.values, forward_all(model, T_IDX["test_balanced"]))}
        if seed == SEEDS[0]:
            torch.save(model.state_dict(), os.path.join(OUT, "models", name + ".pt"))
        json.dump(res, open(path, "w"), indent=1)
        print(f"{name}: val={res['val']:.3f}  marker-balanced={res['marker_test_balanced']:.3f}  "
              f"marker-inverted={res['marker_test_inverted']:.3f}  no-marker={res['clean_test_balanced']:.3f}  "
              f"({time.time() - t0:.0f}s)")

run_marker()

In [ ]:
def gradcam_batch(model, x):
    model.eval()
    feats = model.features(x)
    feats.retain_grad()
    pooled = F.adaptive_avg_pool2d(F.relu(feats), 1).flatten(1)
    logit = model.classifier(pooled).squeeze(1)
    model.zero_grad()
    logit.sum().backward()
    w = feats.grad.mean(dim=(2, 3), keepdim=True)
    cam = F.relu((w * feats).sum(1, keepdim=True)).detach()
    cam = F.interpolate(cam, size=(IMG, IMG), mode="bilinear", align_corners=False)[:, 0]
    cam = cam / (cam.sum(dim=(1, 2), keepdim=True) + 1e-8)          
    return cam.cpu().numpy(), torch.sigmoid(logit).detach().cpu().numpy()

def load_model(path):
    m = new_model()
    m.load_state_dict(torch.load(path, map_location=device, weights_only=True))
    return m.eval()

def cams_for(model, df, bs=16):
    idx = to_idx(df)
    cams, probs = [], []
    for i in range(0, len(idx), bs):
        b = idx[i:i + bs]
        x = make_batch(b, marker=torch.ones(len(b), dtype=torch.bool, device=device))
        c, p = gradcam_batch(model, x)
        cams.append(c)
        probs.append(p)
    return np.concatenate(cams), np.concatenate(probs)

PAD = 16   
HIT_AREA = (min(R1 + PAD, IMG) - max(R0 - PAD, 0)) * (min(C1 + PAD, IMG) - max(C0 - PAD, 0)) / (IMG * IMG)

def peak_hit(cams):
    rr, cc = np.divmod(cams.reshape(len(cams), -1).argmax(1), IMG)
    return float(((rr >= R0 - PAD) & (rr < R1 + PAD) & (cc >= C0 - PAD) & (cc < C1 + PAD)).mean())

age_model = load_model(os.path.join(OUT, "models", "age_f90_s0.pt"))
mk_model = load_model(os.path.join(OUT, "models", "marker_s0.pt"))
sub = pd.concat([test_bal[test_bal.label == l].sample(n=min(40, (test_bal.label == l).sum()), random_state=0)
                 for l in (0, 1)]).reset_index(drop=True)
cam_mk, p_mk = cams_for(mk_model, sub)
cam_ag, p_ag = cams_for(age_model, sub)

area = (R1 - R0) * (C1 - C0) / (IMG * IMG)
mass = lambda c: c[:, R0:R1, C0:C1].sum(axis=(1, 2))
enr_mk, enr_ag = mass(cam_mk) / area, mass(cam_ag) / area
print(f"Grad-CAM mass inside the marker box on {len(sub)} test images (1.0x = chance, box covers {area:.1%} of the image)")
print(f"  marker-trained model : {mass(cam_mk).mean():.3f}  = {enr_mk.mean():.1f}x chance")
print(f"  age-skew model       : {mass(cam_ag).mean():.3f}  = {enr_ag.mean():.1f}x chance")
print(f"  hottest Grad-CAM pixel inside the box (+{PAD}px margin = {HIT_AREA:.1%} of the image, i.e. chance): "
      f"marker model {peak_hit(cam_mk):.0%} | age model {peak_hit(cam_ag):.0%}")
json.dump({"marker_model_enrichment": float(enr_mk.mean()), "age_model_enrichment": float(enr_ag.mean()),
           "marker_model_peak_hit": peak_hit(cam_mk), "age_model_peak_hit": peak_hit(cam_ag), "peak_hit_chance": HIT_AREA,
           "box_area_fraction": float(area), "n_images": int(len(sub))},
          open(os.path.join(OUT, "runs", "gradcam_marker.json"), "w"), indent=1)

def draw(ax, img, cam, title):
    ax.imshow(img, cmap="gray")
    if cam is not None:
        ax.imshow(cam / (cam.max() + 1e-8), cmap="jet", alpha=0.45)
    ax.add_patch(plt.Rectangle((C0, R0), C1 - C0, R1 - R0, fill=False, edgecolor="white", linewidth=1.2))
    ax.set_title(title, fontsize=8)
    ax.axis("off")

rows = list(sub[sub.label == 1].head(2).index) + list(sub[sub.label == 0].head(2).index)
fig, axes = plt.subplots(len(rows), 3, figsize=(7.5, 2.5 * len(rows)))
for r, i in enumerate(rows):
    img = arr[row_of[sub.loc[i, "Image Index"]]].copy()
    img[R0:R1, C0:C1] = 255
    kind = "diseased" if sub.loc[i, "label"] == 1 else "healthy"
    draw(axes[r, 0], img, None, f"{kind}, marker stamped")
    draw(axes[r, 1], img, cam_mk[i], f"marker model (score {p_mk[i]:.2f})")
    draw(axes[r, 2], img, cam_ag[i], f"age-skew model (score {p_ag[i]:.2f})")
fig.suptitle("Grad-CAM with a planted marker vs. with an age shortcut (same inputs)", fontsize=9)
fig.tight_layout()
fig.savefig(os.path.join(OUT, "figures", "fig3_marker_vs_age_gradcam.png"), dpi=140, bbox_inches="tight")
if not TEST:
    plt.show()

In [ ]:
def load_runs(pattern):
    rows = [json.load(open(p)) for p in sorted(glob.glob(os.path.join(OUT, "runs", pattern)))]
    return pd.DataFrame(rows)

df = load_runs("age_f*_s*.json")
ctrl = load_runs("imagenet_s*.json")
mkr = load_runs("marker_s*.json")
df.to_csv(os.path.join(OUT, "sweep_results.csv"), index=False)
cols = ["val", "test_balanced", "test_inverted", "gap_healthy",
        "repair_test_balanced", "repair_test_inverted", "repair_gap_healthy"]
print("MEAN OVER SEEDS, by skew level f (share of positives that are old):")
print(df.groupby("f")[cols].mean().round(3).to_string())
print("\nSTD OVER SEEDS:")
print(df.groupby("f")[cols].std().round(3).to_string())
print("\nImageNet features + balanced head (control):")
print(ctrl[["test_balanced", "test_inverted", "gap_healthy"]].mean().round(3).to_string())
print("\nPlanted-marker control (means):")
print(mkr[["val", "marker_test_balanced", "marker_test_inverted", "clean_test_balanced"]].mean().round(3).to_string())

def show(fig, name):
    fig.tight_layout()
    fig.savefig(os.path.join(OUT, "figures", name), dpi=140, bbox_inches="tight")
    if not TEST:
        plt.show()
    plt.close(fig)

fig, ax = plt.subplots(figsize=(6.4, 3.8))
w = 0.36
for k, (f, col, lab) in enumerate([(0.5, "tab:blue", "trained age-balanced (50/50)"),
                                   (0.9, "tab:red", "trained age-skewed (90/10)")]):
    s = df[np.isclose(df.f, f)]
    vals = [s[c].values for c in ("val", "test_balanced", "test_inverted")]
    x = np.arange(3) + (k - 0.5) * w
    ax.bar(x, [v.mean() for v in vals], w, color=col, alpha=0.75, label=lab)
    for j, v in enumerate(vals):
        ax.scatter(np.full(len(v), x[j]), v, color="black", s=10, zorder=3)
ax.axhline(0.5, ls="--", color="gray", lw=1)
ax.set_xticks(range(3))
ax.set_xticklabels(["validation\n(natural age mix)", "test: balanced", "test: inverted"])
ax.set_ylabel("AUROC (dots = seeds)")
ax.set_ylim(0.3, 1.0)
ax.legend(fontsize=8, loc="lower left")
ax.set_title("Validation vs. test AUROC: age-balanced vs. age-skewed training", fontsize=9)
show(fig, "fig1_validation_hides_collapse.png")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
ax = axes[0]
ax.axvspan(0.50, 0.55, color="gold", alpha=0.35)
ax.text(0.52, 0.31, "natural NIH\nassociations\n(age ~0.50, view ~0.52,\nsex ~0.55)", fontsize=6.5, ha="center", va="bottom")
for c, col, lab in [("test_balanced", "tab:green", "test: balanced"), ("test_inverted", "tab:red", "test: inverted")]:
    ax.scatter(df.f, df[c], s=14, color=col, alpha=0.45)
    m = df.groupby("f")[c].mean()
    ax.plot(m.index, m.values, "-o", color=col, label=lab)
ax.axhline(0.5, ls="--", color="gray", lw=1)
ax.set_xlabel("shortcut strength = AUROC achievable from age alone")
ax.set_ylabel("AUROC of the fine-tuned model")
ax.set_ylim(0.3, 1.0)
ax.legend(fontsize=8)
ax.set_title("AUROC vs. shortcut strength", fontsize=9)
ax = axes[1]
for c, col, lab in [("gap_healthy", "tab:purple", "healthy patients"), ("gap_diseased", "tab:orange", "diseased patients")]:
    ax.scatter(df.f, df[c], s=14, color=col, alpha=0.45)
    m = df.groupby("f")[c].mean()
    ax.plot(m.index, m.values, "-o", color=col, label=lab)
ax.axhline(0, ls="--", color="gray", lw=1)
ax.set_xlabel("shortcut strength")
ax.set_ylabel("mean score: old minus young patients")
ax.legend(fontsize=8)
ax.set_title("Age gap in predicted scores vs. shortcut strength", fontsize=9)
show(fig, "fig2_dose_response.png")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
ax = axes[0]
for c, col, lab in [("test_inverted", "tab:red", "fine-tuned as trained"),
                    ("repair_test_inverted", "tab:blue", "last layer re-fitted on age-balanced data")]:
    ax.scatter(df.f, df[c], s=14, color=col, alpha=0.45)
    m = df.groupby("f")[c].mean()
    ax.plot(m.index, m.values, "-o", color=col, label=lab)
if len(ctrl):
    ax.axhline(ctrl["test_inverted"].mean(), ls=":", color="gray", label="ImageNet features + balanced head")
ax.axhline(0.5, ls="--", color="gray", lw=1)
ax.set_xlabel("shortcut strength")
ax.set_ylabel("AUROC on the inverted test set")
ax.set_ylim(0.3, 1.0)
ax.legend(fontsize=7)
ax.set_title("Inverted-test AUROC: as trained vs. last layer re-fitted", fontsize=9)
ax = axes[1]
for c, col, lab in [("gap_healthy", "tab:red", "as trained"), ("repair_gap_healthy", "tab:blue", "after repair")]:
    ax.scatter(df.f, df[c], s=14, color=col, alpha=0.45)
    m = df.groupby("f")[c].mean()
    ax.plot(m.index, m.values, "-o", color=col, label=lab)
ax.axhline(0, ls="--", color="gray", lw=1)
ax.set_xlabel("shortcut strength")
ax.set_ylabel("healthy patients: score old minus young")
ax.legend(fontsize=8)
ax.set_title("Age gap before vs. after re-fitting the last layer", fontsize=9)
show(fig, "fig4_repair_across_strengths.png")
print("\nfigures saved in", os.path.join(OUT, "figures"), "(figure 3 came from the previous cell)")

In [ ]:
dl = os.path.join(OUT, "deliverables")
shutil.rmtree(dl, ignore_errors=True)
os.makedirs(dl)
for sub_dir in ("runs", "figures", "models"):
    shutil.copytree(os.path.join(OUT, sub_dir), os.path.join(dl, sub_dir))
shutil.copy(os.path.join(OUT, "sweep_results.csv"), dl)
zip_path = shutil.make_archive(os.path.join(OUT, "sweep_outputs"), "zip", dl)
print("saved:", zip_path, f"({os.path.getsize(zip_path) / 1e6:.0f} MB)")

In [ ]:
from scipy.stats import spearmanr

def age_probe_auroc(model, seed):
    tr_df = make_train(0.5, seed)
    probe = fit_head(forward_all(model, to_idx(tr_df), feats=True), (tr_df.age_bracket == "old").astype(int).values)
    p = probe.predict_proba(forward_all(model, T_IDX["test_balanced"], feats=True))[:, 1]
    return auc((test_bal.age_bracket == "old").astype(int).values, p)

def run_probe_sweep():
    for seed in SEEDS:
        path = os.path.join(OUT, "runs", f"probe_imagenet_s{seed}.json")
        if not os.path.exists(path):                       
            torch.manual_seed(seed)
            json.dump({"seed": seed, "age_probe_auroc": age_probe_auroc(new_model(), seed)}, open(path, "w"))
        for f in SKEWS:
            name = f"probe_f{int(round(f * 100))}_s{seed}"
            path = os.path.join(OUT, "runs", name + ".json")
            if os.path.exists(path):
                print("skip (done):", name)
                continue
            t0 = time.time()
            model, _ = train_model(make_train(f, seed), seed)
            p_bal = forward_all(model, T_IDX["test_balanced"])
            p_inv = forward_all(model, T_IDX["test_inverted"])
            res = {"f": f, "seed": seed, "age_probe_auroc": age_probe_auroc(model, seed),
                   "test_balanced": auc(test_bal.label.values, p_bal), "test_inverted": auc(test_inv.label.values, p_inv)}
            res.update(age_gaps(test_bal, p_bal))
            json.dump(res, open(path, "w"), indent=1)
            print(f"{name}: age-probe={res['age_probe_auroc']:.3f}  gap={res['gap_healthy']:.3f}  inverted={res['test_inverted']:.3f}  ({time.time() - t0:.0f}s)")

run_probe_sweep()

def load_probe(pattern):
    return pd.DataFrame([json.load(open(p)) for p in sorted(glob.glob(os.path.join(OUT, "runs", pattern)))])
pr, floor = load_probe("probe_f*_s*.json"), load_probe("probe_imagenet_s*.json")
pr.to_csv(os.path.join(OUT, "probe_results.csv"), index=False)

print("\nMEAN OVER SEEDS by skew level:")
print(pr.groupby("f")[["age_probe_auroc", "gap_healthy", "test_inverted"]].mean().round(3).to_string())
print(f"untuned ImageNet features, age-probe AUROC: {floor.age_probe_auroc.mean():.3f}")
rho = {}
if len(pr) >= 4:
    for col in ("gap_healthy", "test_inverted"):
        rho[col], pval = spearmanr(pr.age_probe_auroc, pr[col])
        print(f"Spearman, age-probe AUROC vs {col}: rho = {rho[col]:.2f} (p = {pval:.3f}, n = {len(pr)} models)")
ref = load_probe("age_f*_s*.json")
if len(ref):                                               
    mm = pr.merge(ref[["f", "seed", "test_inverted"]], on=["f", "seed"], suffixes=("", "_sweep"))
    if len(mm):
        dd = (mm.test_inverted - mm.test_inverted_sweep).abs()
        print(f"rerun vs main sweep with the same seeds: inverted AUROC differs by {dd.mean():.3f} on average (max {dd.max():.3f})")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
ax = axes[0]
ax.scatter(pr.f, pr.age_probe_auroc, s=14, color="tab:blue", alpha=0.45)
mean_by_f = pr.groupby("f").age_probe_auroc.mean()
ax.plot(mean_by_f.index, mean_by_f.values, "-o", color="tab:blue", label="fine-tuned models")
ax.axhline(floor.age_probe_auroc.mean(), ls=":", color="gray", label="untuned ImageNet features")
ax.axhline(0.5, ls="--", color="gray", lw=1)
ax.set_xlabel("shortcut strength")
ax.set_ylabel("AUROC of a probe that reads age from features")
ax.set_title("How readable is age from the features?", fontsize=9)
ax.legend(fontsize=8)
ax = axes[1]
sc = ax.scatter(pr.age_probe_auroc, pr.gap_healthy, c=pr.f, cmap="viridis", s=30)
plt.colorbar(sc, ax=ax, label="shortcut strength")
ax.set_xlabel("age-probe AUROC (readable age)")
ax.set_ylabel("score gap, old minus young (healthy)")
ax.set_title("Readable age vs. actual reliance on age (one dot per model)" + (f"; rho = {rho['gap_healthy']:.2f}" if rho else ""), fontsize=8)
fig.tight_layout()
fig.savefig(os.path.join(OUT, "figures", "fig5_age_probe.png"), dpi=140, bbox_inches="tight")
if not TEST:
    plt.show()
plt.close(fig)

In [ ]:
dl = os.path.join(OUT, "deliverables")
shutil.rmtree(dl, ignore_errors=True)
os.makedirs(dl)
for sub_dir in ("runs", "figures", "models"):
    shutil.copytree(os.path.join(OUT, sub_dir), os.path.join(dl, sub_dir))
for extra in ("sweep_results.csv", "probe_results.csv"):
    if os.path.exists(os.path.join(OUT, extra)):
        shutil.copy(os.path.join(OUT, extra), dl)
zip_path = shutil.make_archive(os.path.join(OUT, "sweep_outputs"), "zip", dl)
print("saved:", zip_path, f"({os.path.getsize(zip_path) / 1e6:.0f} MB)")